# M3 on Kaggle — full-pool edge baselines (E1 centralized + static + BLE)

**Inputs:** attach your private dataset containing the M1 `processed/` folder
(`data_manifest.json`, `wifi_mqtt_full.parquet`, `wifi_profiling.parquet`, `ble.parquet`, ...).
**Settings:** Internet *On*. Accelerator: *None* is fine; the XGBoost GPU option is in cell 3.
**Run:** *Save Version → Save & Run All (Commit)* so it keeps running with the browser closed.
Expected runtime with 4 vCPU is roughly 1–2 h (the t5 model trains on ~7.3M rows).

**Output:** `/kaggle/working/m3_full_outputs.zip` contains `reports/m3/full/`, `results.jsonl` and the models.
Download it and unzip at the repo root. The partition is rebuilt here, and it is identical to the local one.

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full

In [ ]:
# M3 full-pool run. With a GPU attached you can add:  --set models.xgb.device=cuda
!python scripts/m3_baselines.py --env kaggle --pool full

In [ ]:
import shutil, glob, os
runs = sorted(glob.glob("/kaggle/working/results/runs/*_m3_baselines_full"))
stage = "/kaggle/working/m3_stage"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree("reports/m3/full", f"{stage}/reports/m3/full")
if runs:
    shutil.copytree(runs[-1], f"{stage}/results/runs/{os.path.basename(runs[-1])}")
shutil.make_archive("/kaggle/working/m3_full_outputs", "zip", stage)
print(open("reports/m3/full/RESULTS.md").read()[:3000])